## ✈️ Flight Price Prediction

This notebook predicts Indian domestic flight ticket prices using a **Random Forest Regressor** trained on features like airline, route, departure time, and number of stops.

**Dataset:** Data_Train.xlsx (10,683 rows) + Test_set.xlsx (2,671 rows)

In [ ]:
import io
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load datasets
df_train = pd.read_excel("Data_Train.xlsx")
df_test  = pd.read_excel("Test_set.xlsx")

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)
df_train.head()


## Load & Combine Data

In [ ]:
# Combine train and test
big_df = pd.concat([df_train, df_test], sort=False)
print("Combined shape:", big_df.shape)
big_df.head()


In [ ]:
big_df.dtypes

##  Feature Engineering
### Date Features

In [ ]:
# Extract Date, Month, Year from Date_of_Journey
big_df["Date"]  = big_df["Date_of_Journey"].str.split("/").str[0]
big_df["Month"] = big_df["Date_of_Journey"].str.split("/").str[1]
big_df["Year"]  = big_df["Date_of_Journey"].str.split("/").str[2]
big_df = big_df.drop("Date_of_Journey", axis="columns")
big_df["Date"]  = big_df["Date"].astype(int)
big_df["Month"] = big_df["Month"].astype(int)
big_df["Year"]  = big_df["Year"].astype(int)


### Arrival Time Cleanup

In [ ]:
# Remove date suffix from Arrival_Time (e.g. '17:00 18 Mar')
big_df["Arrival_Time"] = big_df["Arrival_Time"].str.split(" ").str[0]


### Null Value Check

In [ ]:
big_df.isnull().sum()

In [ ]:
big_df[big_df["Total_Stops"].isnull()]


In [ ]:
big_df["Total_Stops"] = big_df["Total_Stops"].fillna("1 stop")
big_df["Total_Stops"] = big_df["Total_Stops"].replace("non-stop", "0 stop")
big_df["Total_Stops"].unique()


### Stops Feature

In [ ]:
big_df["Stop"] = big_df["Total_Stops"].str.split(" ").str[0]
big_df = big_df.drop("Total_Stops", axis="columns")


### Departure & Arrival Time Features
> **Bug fix:** Original code used `.str[0]` for both `Dep_Hour` and `Dep_Min`. Fixed `Dep_Min` to use `.str[1]`.

In [ ]:
# Extract Departure Hour and Minute  (Bug fix: original code used str[0] for both)
big_df["Dep_Hour"] = big_df["Dep_Time"].str.split(":").str[0].astype(int)
big_df["Dep_Min"]  = big_df["Dep_Time"].str.split(":").str[1].astype(int)
big_df = big_df.drop("Dep_Time", axis="columns")


In [ ]:
big_df["Arrival_Hour"]   = big_df["Arrival_Time"].str.split(":").str[0].astype(int)
big_df["Arrival_Minute"] = big_df["Arrival_Time"].str.split(":").str[1].astype(int)
big_df = big_df.drop("Arrival_Time", axis="columns")


### Route Features

In [ ]:
for i in range(5):
    big_df[f"Route_{i+1}"] = big_df["Route"].str.split("→ ").str[i]
for i in range(1, 6):
    big_df[f"Route_{i}"].fillna("None", inplace=True)
big_df = big_df.drop("Route", axis="columns")


### Price Fill & Drop Duration

In [ ]:
big_df["Price"].fillna(big_df["Price"].mean(), inplace=True)
big_df = big_df.drop("Duration", axis=1)
big_df.head()


## Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
cat_cols = ["Airline", "Destination", "Additional_Info", "Source",
            "Route_1", "Route_2", "Route_3", "Route_4", "Route_5", "Stop"]
for col in cat_cols:
    big_df[col] = le.fit_transform(big_df[col].astype(str))
print("Shape after encoding:", big_df.shape)
big_df.head()


## Train / Test Split

In [ ]:
train_data = big_df.iloc[:10683]
test_data  = big_df.iloc[10683:]

X = train_data.drop("Price", axis=1)
y = train_data["Price"]
print("X shape:", X.shape, "| y shape:", y.shape)


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
print("X_train:", X_train.shape, "| X_test:", X_test.shape)


## Feature Selection (Lasso)

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.feature_selection import SelectFromModel

fs_model = SelectFromModel(Lasso(alpha=0.05, random_state=0))
fs_model.fit(X_train, y_train)
selected_features = X_train.columns[fs_model.get_support()]
print("Selected features:", selected_features.tolist())


In [ ]:
# Year has low variance/importance - drop as per feature selection
X_train = X_train.drop("Year", axis=1)
X_test  = X_test.drop("Year", axis=1)


## Model Training — Random Forest

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

n_estimators      = [int(x) for x in np.linspace(start=100, stop=1200, num=12)]
max_features      = ["sqrt"]   # 'auto' removed in sklearn 1.2+; 'sqrt' is equivalent for regressors
max_depth         = [int(x) for x in np.linspace(start=5, stop=30, num=6)]
min_samples_split = [2, 5, 10, 15, 100]
min_samples_leaf  = [1, 2, 5, 10]

random_grid = {
    "n_estimators":      n_estimators,
    "max_features":      max_features,
    "max_depth":         max_depth,
    "min_samples_split": min_samples_split,
    "min_samples_leaf":  min_samples_leaf
}
print(random_grid)


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor()
rf_random = RandomizedSearchCV(
    estimator=rf,
    param_distributions=random_grid,
    n_iter=50,
    scoring='neg_mean_squared_error',
    cv=2,
    verbose=2,
    n_jobs=-1,
    random_state=42
)
rf_random.fit(X_train, y_train)
print("Best params:", rf_random.best_params_)


##  Model Evaluation

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = rf_random.predict(X_test)

r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"R² Score : {r2:.4f}")
print(f"MAE      : {mae:.2f}")
print(f"MSE      : {mse:.2f}")
print(f"RMSE     : {rmse:.2f}")


## Visualisations
### Actual vs Predicted

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.4, color='steelblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel("Actual Price (₹)")
plt.ylabel("Predicted Price (₹)")
plt.title("Actual vs Predicted Flight Prices")
plt.tight_layout()
plt.show()


### Residuals Distribution

In [ ]:
import seaborn as sns

residuals = y_test - y_pred
sns.histplot(residuals, kde=True, color='coral')
plt.title("Residuals Distribution")
plt.xlabel("Residual (Actual - Predicted)")
plt.tight_layout()
plt.show()
